In [1]:
from ultralytics import YOLO
import cv2

#loading model
model = YOLO('../models/best.pt')

# running on test image 
results = model('../data/test_car.jpg', conf=0.5)

results[0].show()

print(f"Detected: {len(results[0].boxes)} plates")


image 1/1 d:\Projects\anpr-system\notebooks\..\data\test_car.jpg: 352x640 2 License_Plates, 265.7ms
Speed: 19.4ms preprocess, 265.7ms inference, 22.6ms postprocess per image at shape (1, 3, 352, 640)
Detected: 2 plates


In [2]:
!pip install easyocr


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import easyocr
print("EasyOCR ready!")

EasyOCR ready!


In [4]:
from ultralytics import YOLO
import easyocr
import cv2
import numpy as np

model = YOLO('../models/best.pt')
reader = easyocr.Reader(['en'])


image_path = '../data/test_car.jpg'
results = model(image_path, conf=0.5)

image = cv2.imread(image_path)

def preprocess_plate(plate_img):

    #inc size of the image for clear vision
    plate_img = cv2.resize(plate_img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # Grayscale
    gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)

    #increasing contrast
    gray = cv2.equalizeHist(gray)

    #removing blur and noise
    gray = cv2.GaussianBlur(gray, (3,3), 0)

    #binary image
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh

for box in results[0].boxes:

    #finding bounding box coordinates
    x1, y1, x2, y2 = map(int, box.xyxy[0])

    # croping plate from full image
    plate_crop = image[y1:y2, x1:x2]

    #preprocess
    processed = preprocess_plate(plate_crop)

    # running ocr on cropped plate
    ocr_result = reader.readtext(processed, detail=1, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

    print(f"Detected box: ({x1}, {y1}) to ({x2}, {y2})")
    for (bbox, text, confidence) in ocr_result:
        print(f"Plate Text: {text}")
        print(f"Confidence: {confidence:.2f}")

    cv2.imshow('Cropped Plate', plate_crop)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



image 1/1 d:\Projects\anpr-system\notebooks\..\data\test_car.jpg: 352x640 2 License_Plates, 215.1ms
Speed: 6.8ms preprocess, 215.1ms inference, 3.8ms postprocess per image at shape (1, 3, 352, 640)


d:\Projects\anpr-system\anpr-env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Detected box: (48, 37) to (268, 86)
Plate Text: NHTZDE1433
Confidence: 0.57
Detected box: (48, 42) to (220, 89)
Plate Text: MHTZDE14
Confidence: 0.47


In [5]:
import re

def fix_plate_text(text):
    
    fixes = {
        'O': '0',   # Letter O → Zero
        'I': '1',   # Letter I → One
        'Z': '2',   # Z → 2
        'S': '5',   # S → 5
        'B': '8',   # B → 8
    }
    
    text = text.upper().replace(' ', '')
    
    # Indian plate pattern: 2 letters + 2 numbers + 2 letters + 4 numbers
    # Pehle 2 chars = state code = letters 
    # Next 2 chars = district = numbers 
    
    result = ''
    for i, char in enumerate(text):
        if i < 2:
            # State code — letter hona chahiye
            result += char if char.isalpha() else list(fixes.keys())[list(fixes.values()).index(char)] if char in fixes.values() else char
        elif i < 4:
            # District number — digit hona chahiye  
            result += fixes.get(char, char) if char.isalpha() else char
        elif i < 6:
            # Series — letter hona chahiye
            result += char if char.isalpha() else char
        else:
            # Number — digit hona chahiye
            result += fixes.get(char, char) if char.isalpha() else char
    
    return result

# Test karo
raw = "NHTZDE1433"
corrected = fix_plate_text(raw)
print(f"Raw OCR    : {raw}")
print(f"Corrected  : {corrected}")
print(f"Actual     : MH12DE1433")

Raw OCR    : NHTZDE1433
Corrected  : NHT2DE1433
Actual     : MH12DE1433


In [6]:
def fix_plate_text(text):
    text = text.upper().replace(' ', '')
    
    # Common OCR mistakes
    char_fixes = {
        'O': '0', 'I': '1', 'Z': '2',
        'S': '5', 'B': '8', 'G': '6',
        'T': '1',  # T → 1 (tera case)
        'N': 'M',  # N → M (tera case)  
    }
    
    result = ''
    for i, char in enumerate(text):
        if i < 2 or (i >= 4 and i < 6):
            # State code aur series = letters rehne do
            result += char
        else:
            # District number aur plate number = digits hone chahiye
            result += char_fixes.get(char, char) if char.isalpha() else char
    
    return result

raw = "NHTZDE1433"
print(f"Corrected : {fix_plate_text(raw)}")

Corrected : NH12DE1433


In [9]:
from ultralytics import YOLO
import easyocr
import cv2
import numpy as np

model = YOLO('../models/best.pt')
reader = easyocr.Reader(['en'])

def preprocess_plate(img):
    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    gray = cv2.GaussianBlur(gray, (3,3), 0)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh


def fix_plate(text):
    text = text.upper().replace(' ','')
    fixes = {'O':'0', 'I':'1', 'Z':'2','S':'5','B':'8','T':'1','N':'M'}
    result = ''
    for i, c in enumerate(text):
        if i < 2 or 4 <= i < 6:
            result += c # letters zone
        else:
            result += fixes.get(c,c) if c.isalpha() else c # digits zone

    return result


def run_anpr(image_path):
    image = cv2.imread(image_path)
    results = model(image_path, conf=0.5)

    plates = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        crop = image[y1:y2, x1:x2]
        processed = preprocess_plate(crop)
        ocr_out = reader.readtext(processed,  allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
        
        raw_text = ' '.join([t for _, t, _ in ocr_out])
        fixed_text = fix_plate(raw_text)
        

        plates.append({
            'bbox': (x1, y1, x2, y2),
            'detection_conf': round(conf, 2),
            'raw_ocr': raw_text,
            'plate_text': fixed_text
        })
    
    return image, plates

print("Pipeline ready!")


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Pipeline ready!


In [10]:
images, plates = run_anpr('../data/test_car.jpg')

for i, p in enumerate(plates):
    print(f"---Plate {i+1} ---")
    print(f"Detection confidence : {p['detection_conf']}")
    print(f"Raw OCR              : {p['raw_ocr']}")
    print(f"Final plate text     : {p['plate_text']}")


image 1/1 d:\Projects\anpr-system\notebooks\..\data\test_car.jpg: 352x640 2 License_Plates, 159.8ms
Speed: 3.6ms preprocess, 159.8ms inference, 5.5ms postprocess per image at shape (1, 3, 352, 640)
---Plate 1 ---
Detection confidence : 0.86
Raw OCR              : NHTZDE1433
Final plate text     : NH12DE1433
---Plate 2 ---
Detection confidence : 0.76
Raw OCR              : MHTZDE14
Final plate text     : MH12DE14


In [11]:
for p in plates:
    x1, y1, x2, y2 = p['bbox']
    # Bounding box 
    cv2.rectangle(image, (x1,y1), (x2,y2), (0,255,0), 2)
    # Plate text 
    cv2.putText(image, p['plate_text'], (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

cv2.imwrite('../data/output.jpg', image)
print("Saved → data/output.jpg")

Saved → data/output.jpg
